# Importation

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import pandas as pd
import re
import numpy as np
import shutil
if not hasattr(np, 'unicode_'):
    np.unicode_ = str
import ast
import os
import math
import plotly.express as px
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, Tuple, List
from scipy.signal import chirp, find_peaks, peak_widths

In [6]:
cosmicai_path = 'data'

df = pd.read_parquet(f'{cosmicai_path}/cleaned_qa2_dataset_scored.parquet')
df.reset_index(drop=True, inplace=True)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38881 entries, 0 to 38880
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   eb_uid                    38881 non-null  object 
 1   time                      38881 non-null  float64
 2   receiver_band             38881 non-null  object 
 3   ref_antenna_name          38881 non-null  object 
 4   antenna_name              38881 non-null  object 
 5   baseband_name             38881 non-null  object 
 6   spw_name_ms               38881 non-null  object 
 7   pol_id                    38881 non-null  object 
 8   frequency_array           38881 non-null  object 
 9   amplitude                 38881 non-null  object 
 10  phase                     38881 non-null  object 
 11  flag_array                38881 non-null  object 
 12  QA2 Flag(s)               38881 non-null  object 
 13  atmospheric_interference  38881 non-null  object 
 14  score_

In [13]:
import pandas as pd

df1 = pd.read_parquet(f'{cosmicai_path}/QA2_WithoutFlags.parquet')
df2 = pd.read_parquet(f'{cosmicai_path}/QA2_WithFlags.parquet')

# Basic check: are they identical?
match = df1["rank_score"].reset_index(drop=True).equals(
        df2["rank_score"].reset_index(drop=True))
print(f"Identical: {match}")

# If not identical, how many differ?
if not match:
    diff = (df1["rank_score"].reset_index(drop=True) -
            df2["rank_score"].reset_index(drop=True)).abs()
    print(f"Rows with any difference:    {(diff > 0).sum()}")
    print(f"Rows with difference > 1e-6: {(diff > 1e-6).sum()}")
    print(f"Max difference: {diff.max():.6e}")
    print(f"Mean difference: {diff.mean():.6e}")

Identical: False
Rows with any difference:    2146
Rows with difference > 1e-6: 1860
Max difference: 9.995713e-01
Mean difference: 1.717297e-02


In [ ]:
def _to_np(x):
    a = np.asarray(x, dtype=float)
    return a[np.isfinite(a)]

def classify_row(
    amplitude,
    min_len=16,
    abs_std_thresh=1e-2,
    edge_frac=0.03,
    k_min=4,
    z_edge=10.0,
    rel_edge=0.10,
    low_var_cv_thresh=1e-4,
    rolloff_frac=0.15,
    rolloff_drop_thresh=0.03,
):
    y = _to_np(amplitude)
    n = y.size
    if n < min_len:
        return "too_short"

    mu = y.mean()
    max_dev = np.max(np.abs(y - mu))
    if max_dev < abs_std_thresh:
        cv = np.std(y) / max(abs(mu), 1e-15)
        if cv < low_var_cv_thresh:
            return "low_var"

    k = max(int(edge_frac * n), k_min)
    k = min(k, n // 4)
    if k >= 2:
        left_mean  = float(np.mean(y[:k]))
        right_mean = float(np.mean(y[-k:]))
        center     = y[k:-k] if n >= 2 * k + 1 else y
        c_med      = float(np.median(center))
        c_std      = float(np.std(center))
        thr        = max(z_edge * c_std, rel_edge * max(abs(c_med), 1e-9))
        left_dev   = left_mean  - c_med
        right_dev  = right_mean - c_med
        if (abs(left_dev) > thr
                and abs(right_dev) > thr
                and np.sign(left_dev) == np.sign(right_dev)):
            return "edge_plateau"

    rolloff_k = max(int(rolloff_frac * n), k_min * 2)
    rolloff_k = min(rolloff_k, n // 4)
    if rolloff_k >= 4:
        center     = y[rolloff_k:-rolloff_k] if n > 2 * rolloff_k else y
        c_med      = float(np.median(center))
        c_level    = max(abs(c_med), 1e-9)
        right_drop = (c_med - float(np.mean(y[-rolloff_k:]))) / c_level
        left_drop  = (c_med - float(np.mean(y[:rolloff_k])))  / c_level
        if right_drop > rolloff_drop_thresh or left_drop > rolloff_drop_thresh:
            return "edge_rolloff"

    return "regular"

# --- Preserve index as uid before any filtering ---
df["uid"] = df.index.copy()

# --- Filter: length < 128 ---
df["_amp_len"] = df["amplitude"].apply(lambda x: len(np.asarray(x)))
before = len(df)
df = df[df["_amp_len"] >= 128].drop(columns=["_amp_len"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with length < 128  →  {len(df)} remaining")

# --- Classify ---
df["_row_type"] = df["amplitude"].apply(classify_row)
counts = df["_row_type"].value_counts()
print(f"\nClassification breakdown:\n{counts.to_string()}")

# --- Filter: keep only regular ---
before = len(df)
df = df[df["_row_type"] == "regular"].drop(columns=["_row_type"]).reset_index(drop=True)
print(f"\nDropped {before - len(df)} bad rows  →  {len(df)} remaining")

In [ ]:
df.columns

In [ ]:
bins = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0]
col  = "rank_score"

print(f"{'Bucket':<18}  {'Count':>6}  {'Pct':>7}")
print("-" * 36)
for hi, lo in zip(bins, bins[1:]):
    count = int(((df[col] > lo) & (df[col] <= hi)).sum())
    print(f"({lo:.1f}, {hi:.1f}]  {count:>6}  {100*count/len(df):>6.1f}%")
print("-" * 36)
print(f"Total  {len(df):>6}  100.0%")
print(f"\nMin: {df[col].min():.4f}  Max: {df[col].max():.4f}  Mean: {df[col].mean():.4f}")

In [ ]:
lo, hi = 0.3, 0.4 
subset = df[(df["rank_score"] > lo) & (df["rank_score"] < hi)].reset_index(drop=True)

print(f"Rows with edge rolloff AND rank_score > 0.4: {len(subset)}")

for _, row in subset.iterrows():
    freq = np.asarray(row["frequency_array"], dtype=float)
    amp  = np.asarray(row["amplitude"],       dtype=float)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(freq, amp, lw=0.8, color="C0")

    a_u = int(row["win_unmasked_start"])
    b_u = int(row["win_unmasked_end"])
    if 0 <= a_u < b_u < len(freq):
        ax.axvspan(freq[a_u], freq[b_u], color="C2", alpha=0.25, label="Window")

    ax.set_title(
        f"score={row['rank_score']:.4f}  |  "
        f"antenna={row['antenna_name']}  |  pol={row['pol_id']}",
        fontsize=8
    )
    ax.set_xlabel("Frequency (GHz)", fontsize=8)
    ax.set_ylabel("Amplitude",       fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# Create label
df["label"] = df["rank_score"] > 0.4

# Start and end: win_masked_start/end for labelled rows, -1 otherwise
df["start"] = df["win_masked_start"].where(df["label"], other=-1)
df["end"]   = df["win_masked_end"].where(df["label"],   other=-1)

# Keep only required columns
keep_cols = [
    "eb_uid", "time", "receiver_band", "ref_antenna_name",
    "antenna_name", "baseband_name", "spw_name_ms", "pol_id",
    "frequency_array", "amplitude", "phase",
    "label", "start", "end"
]
df_out = df[keep_cols].copy()

print(f"Total rows:     {len(df_out)}")
print(f"Label True:     {df_out['label'].sum()}")
print(f"Label False:    {(~df_out['label']).sum()}")

df_out.to_parquet("data/qa2_labelled_dataset.parquet", engine="pyarrow", compression="zstd")
print("Saved: qa2_labelled_dataset.parquet")

In [ ]:
df_csv = pd.read_csv("data/bp_anomalies_indices.csv")
df_pq  = pd.read_parquet("data/cleaned_qa2_dataset_scored.parquet")

key_csv = ["eb_uid", "spw_name_ms", "antenna_name"]
key_pq  = ["eb_uid", "spw_name_ms", "antenna_name"]

for col in key_csv:
    df_csv[col] = df_csv[col].astype(str).str.strip()
for col in key_pq:
    df_pq[col]  = df_pq[col].astype(str).str.strip()

merged = df_pq.merge(
    df_csv[key_csv].drop_duplicates(),
    left_on  = key_pq,
    right_on = key_csv,
    how      = "inner",
)

print(f"CSV rows:             {len(df_csv)}")
print(f"Parquet rows:         {len(df_pq)}")
print(f"Matching rows:        {len(merged)}")
print(f"Unique matching keys: {merged[key_pq].drop_duplicates().shape[0]}")

In [ ]:
merged_sorted = merged.sort_values("rank_score", ascending=True)
print(merged_sorted[["eb_uid", "spw_name_ms", "antenna_name", "pol_id", "rank_score"]].head(5).to_string())

In [ ]:
col = "rank_score"

print(f"Plotting {len(merged_sorted)} rows...")

for _, row in merged_sorted.iterrows():
    freq = np.asarray(row["frequency_array"], dtype=float)
    amp  = np.asarray(row["amplitude"],       dtype=float)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(freq, amp, lw=0.8, color="C0")

    # Window shading
    a_u = int(row["win_unmasked_start"])
    b_u = int(row["win_unmasked_end"])
    if 0 <= a_u < b_u < len(freq):
        ax.axvspan(freq[a_u], freq[b_u], color="C2", alpha=0.25, label="Window")

    ax.set_title(
        f"score={row[col]:.4f}  |  "
        f"eb={row['eb_uid']}  |  spw={row['spw_name_ms']}  |  "
        f"antenna={row['antenna_name']}  |  pol={row['pol_id']}",
        fontsize=7
    )
    ax.set_xlabel("Frequency (GHz)", fontsize=8)
    ax.set_ylabel("Amplitude",       fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

# Dataset Classification

In [ ]:
cosmicai_path = 'Data'

df = pd.read_parquet(f'{cosmicai_path}/QA2_Bandpass_Data_Labeled.parquet')
df.reset_index(drop=True, inplace=True)

In [ ]:
idx = 76844
f = np.asarray(df.loc[idx, "frequency_array"], dtype=np.float64)
d = np.diff(f)

spacing_df = pd.DataFrame({
    "i": np.arange(len(d)),
    "f_i": f[:-1],
    "f_next": f[1:],
    "delta": d,
})
print(spacing_df.head(1000))


In [ ]:
df.columns

In [ ]:
def _to_np(x):
    a = np.asarray(x, dtype=float)
    return a[np.isfinite(a)]

def classify_row(amplitude, min_len=8, abs_std_thresh=1e-6,
                 edge_frac=0.05, k_min=8, z_edge=5.0, rel_edge=0.02):
    y = _to_np(amplitude)
    n = y.size
    if n < min_len:
        return "regular"

    mu = y.mean()
    if np.max(np.abs(y - mu)) < abs_std_thresh:
        return "low_var"

    k = max(int(edge_frac * n), k_min)
    k = min(k, n // 4)
    left_mean  = float(np.mean(y[:k]))
    right_mean = float(np.mean(y[-k:]))
    center     = y[k:-k] if n >= 2 * k + 1 else y
    c_med      = float(np.median(center))
    c_std      = float(np.std(center))
    thr        = max(z_edge * c_std, rel_edge * max(abs(c_med), 1e-9))

    left_dev  = left_mean  - c_med
    right_dev = right_mean - c_med
    if (abs(left_dev) > thr) and (abs(right_dev) > thr) and (np.sign(left_dev) == np.sign(right_dev)):
        return "edge_plateau"

    return "regular"

def _majority_with_tiebreak(s):
    priority = {"edge_plateau": 2, "low_var": 1, "regular": 0}
    vc = s.value_counts()
    best, bestc = None, -1
    for k, c in vc.items():
        if c > bestc or (c == bestc and priority.get(k, -1) > priority.get(best, -1)):
            best, bestc = k, c
    return best

dfC = df.copy()
dfC["row_type"] = dfC["amplitude"].apply(classify_row)

grp_cols = ["eb_uid", "antenna_name", "spw_name_ms", "pol_id"]

grp_majority = (dfC
    .groupby(grp_cols, dropna=False)["row_type"]
    .apply(_majority_with_tiebreak)
    .rename("grp_type")
    .reset_index())

grp_counts = (dfC
    .groupby(grp_cols + ["row_type"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index())

df_labeled = df.merge(grp_majority, on=grp_cols, how="left")

df_lowvar       = df_labeled[df_labeled["grp_type"] == "low_var"].copy()
df_edge_plateau = df_labeled[df_labeled["grp_type"] == "edge_plateau"].copy()
df_regular      = df_labeled[df_labeled["grp_type"] == "regular"].copy()

In [ ]:
df_lowvar.info()

In [ ]:
df_edge_plateau.info()

In [ ]:
df_regular["amplitude_len"] = df_regular["amplitude"].str.len()

grouped_df = df_regular.groupby("amplitude_len")


In [ ]:
grouped_df.size()

In [ ]:
group = grouped_df.get_group(3840)

In [ ]:
row = group.iloc[3307]

freq_ghz = row["frequency_array"] * 1e-9
amp = row["amplitude"]

plt.plot(freq_ghz, amp)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Amplitude")
plt.title("Amplitude vs Frequency of a Bandpass Calibration Having No Anomaly")
plt.tight_layout()
plt.show()


In [ ]:
df_edge_plateau.head(10)

In [ ]:
def make_hashable(x):
    if isinstance(x, np.ndarray):
        return tuple(make_hashable(e) for e in x.tolist())
    if isinstance(x, list):
        return tuple(make_hashable(e) for e in x)
    if isinstance(x, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in x.items()))
    if isinstance(x, (np.generic,)):  # e.g., np.int64, np.float32
        return x.item()
    return x  # str, int, float, None, etc.

hcol = df_edge_plateau["QA2 Flag(s)"].map(make_hashable)

# counts of each unique value (including NaN as its own bucket)
counts = hcol.value_counts(dropna=False)

out = counts.rename_axis("QA2 Flag(s)").reset_index(name="count")
out["pct"] = out["count"] / out["count"].sum()
out


In [ ]:
row = df_edge_plateau.loc[368]  # or df.query("eb_uid == @uid & pol_id == 'X'").iloc[0]

# Parse columns that may be either lists or strings
def to_array(x):
    if isinstance(x, str):   # stringified list
        x = ast.literal_eval(x)
    return np.asarray(x, dtype=float)

freq = to_array(row["frequency_array"])   # Hz in your sample
amp  = to_array(row["amplitude"])

# Optional: convert Hz → GHz to match your earlier plots
freq_GHz = freq / 1e9

plt.figure(figsize=(8, 3))
plt.plot(freq_GHz, amp, lw=1)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Amplitude")
plt.title(f"eb_uid={row['eb_uid']}  ant={row['antenna_name']}  pol={row['pol_id']}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
amp_len = df_lowvar["amplitude"].apply(lambda v: len(v) if hasattr(v, "__len__") else np.nan)
unique_lengths = np.sort(amp_len.dropna().unique())
print(unique_lengths)

In [ ]:
amp_len = df_edge_plateau["amplitude"].apply(lambda v: len(v) if hasattr(v, "__len__") else np.nan)
unique_lengths = np.sort(amp_len.dropna().unique())
print(unique_lengths)

In [ ]:
df_lowvar.to_parquet("qa2_low_variance.parquet", engine='pyarrow', compression="zstd")

In [ ]:
df_edge_plateau.to_parquet("qa2_edge_plateau.parquet", engine='pyarrow', compression="zstd")

# Demonstration

In [ ]:
bandpass_sel_df = df.query('eb_uid == "uid://A002/Xff45a6/X4435"').copy()

In [ ]:
fig1 = px.scatter(
    x = bandpass_sel_df.iloc[0]['frequency_array'],
    y = bandpass_sel_df.iloc[0]['amplitude_corr_tsys'],
    render_mode='webgl', range_y=[0, 0.02], 
    labels={'x': 'Frequency [Hz]', 'y': 'Amplitude [?]'}
)
fig1

In [ ]:
fig1 = px.scatter(
    x = bandpass_sel_df.iloc[20]['frequency_array'],
    y = bandpass_sel_df.iloc[20]['phase_norm_wrap'],
    render_mode='webgl', range_y=[-np.pi, np.pi], 
    labels={'x': 'Frequency [Hz]', 'y': 'Phase [rad]'}
)
fig1

In [ ]:
bandpass_explode_df = bandpass_sel_df.explode(['frequency_array','amplitude_corr_tsys','phase_norm_wrap'])

bandpass_explode_df['label'] = bandpass_explode_df.apply(
    lambda x: f'{x["antenna"]}, {x["spw_name"]}, pol {x["polarization"]}', axis=1)

In [ ]:
fig2 = px.scatter(
    bandpass_explode_df,
    x='frequency_array', 
    y='amplitude_corr_tsys',
    color='label', render_mode='webgl',
)
fig2.update_traces(mode='lines', opacity=0.5)

In [ ]:
fig2 = px.line(
    bandpass_explode_df.query('antenna == ["DV24", "DA46", "DV09"] and spw_name == "SpectralWindow_68"'),
    x='frequency_array', 
    y='amplitude_corr_tsys',
    color='label', render_mode='webgl'
)
fig2

In [ ]:
fig2 = px.line(
    bandpass_explode_df.query('antenna == "DA64"'),
    x='frequency_array', 
    y='amplitude_corr_tsys',
    color='label', render_mode='webgl'
)
fig2

In [ ]:
fig2 = px.line(
    bandpass_explode_df.query('antenna == "DV09"'),
    x='frequency_array', 
    y='amplitude_corr_tsys',
    color='label', render_mode='webgl'
)
fig2

In [ ]:
fig2 = px.line(
    bandpass_explode_df.query('antenna == "DA59"'),
    x='frequency_array', 
    y='amplitude_corr_tsys',
    color='label', render_mode='webgl'
)
fig2

In [ ]:
fig2 = px.line(
    bandpass_explode_df.query('antenna == ["DA61", "DV03"]'),
    x='frequency_array', 
    y='phase_norm_wrap',
    color='label', render_mode='webgl'
)
fig2

# Data Analysis

In [ ]:
actual_specs = [np.array(x, dtype=float)
                 for x in df['amplitude'].tolist()]

In [ ]:
freqs = [np.array(x, dtype=float)
                 for x in df['frequency_array'].tolist()]

In [ ]:
eb_uid = df['eb_uid']
antenna = df['antenna_name']
spw_name_ms = df['spw_name_ms']
polarization = df['pol_id']

In [ ]:
length_groups: Dict[int, List[int]] = {}
for i, s in enumerate(actual_specs):
    L = s.shape[0]
    length_groups.setdefault(L, []).append(i)

In [ ]:
result: Dict[int, Tuple[np.ndarray, ...]] = {}
for L, idxs in length_groups.items():
    actual_specs_L = np.vstack([actual_specs[i] for i in idxs])
    freqs_L = np.vstack([freqs[i] for i in idxs])
    eb_uid_L = eb_uid[idxs]
    antenna_L = antenna[idxs]
    spw_name_ms_L = spw_name_ms[idxs]
    polarization_L = polarization[idxs]
    result[L] = (actual_specs_L, freqs_L, eb_uid_L, antenna_L, spw_name_ms_L, polarization_L)

In [ ]:
print(f"Found lengths: {sorted(result.keys())}")

In [ ]:
lengths, step_means, step_stds, span_means, span_stds = [], [], [], [], []
 
for length in sorted(result):
    actual_specs_L, freqs_L, eb_uid_L, antenna_L, spw_name_ms_L, polarization_L = result[length]
    print('Analyzing channel length of', length)
    lengths.extend(lengths)
    freq_step = abs(freqs_L[:, 1] - freqs_L[:, 0])
    freq_span = abs(freqs_L[:, -1] - freqs_L[:, 0])
    for name, arr in [("Step", freq_step)]:
        print(f"--- {name} ---")
        print("  count:", arr.size)
        print("    mean:    ", np.mean(arr) / 1e6, "Mhz")
        print("    std:     ", np.std(arr) / 1e6, "Mhz")
        print("    min/max: ", np.min(arr) / 1e6, "Mhz", np.max(arr) / 1e6, "Mhz")
        print("    median:  ", np.median(arr) / 1e6, "Mhz")
        print("    25/75%:  ", np.percentile(arr, [25, 75]) / 1e6, "Mhz")
        print()
    for name, arr in [("Span", freq_span)]:
        print(f"--- {name} ---")
        print("  count:", arr.size)
        print("    mean:    ", np.mean(arr) / 1e6, "Mhz")
        print("    std:     ", np.std(arr) / 1e6, "Mhz")
        print("    min/max: ", np.min(arr) / 1e6, "Mhz", np.max(arr) / 1e6, "Mhz")
        print("    median:  ", np.median(arr) / 1e6, "Mhz")
        print("    25/75%:  ", np.percentile(arr, [25, 75]) / 1e6, "Mhz")
        print()

# Experimentation

In [ ]:
trans_df = pd.read_parquet(f'{cosmicai_path}/full_spectrum.gzip')

In [ ]:
df['frequency_array'] = df['frequency_array'] \
    .apply(lambda freqs: [f/1e9 for f in freqs])

In [ ]:
df['frequency_array']

In [ ]:
trans_freqs = trans_df['Frequency (GHz)'].values
trans_vals  = trans_df['Transmission (%)'].values

In [ ]:
def match_and_correct(freq_array):
    idxs = np.searchsorted(trans_freqs, freq_array)
    idxs[idxs == len(trans_freqs)] = len(trans_freqs) - 1
    left  = np.maximum(idxs - 1, 0)
    right = idxs
    dl = np.abs(freq_array - trans_freqs[left])
    dr = np.abs(trans_freqs[right] - freq_array)
    nearest = np.where(dl <= dr, left, right)
    mt = trans_vals[nearest]
    return mt

In [ ]:
results = df.apply(
    lambda row: match_and_correct(
        np.array(row['frequency_array'], dtype=float)
    ),
    axis=1
)

In [ ]:
df['transmission_array'] = results

In [ ]:
plt.figure(figsize=(100, 20))
plt.plot(trans_freqs, trans_vals, marker='o')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Transmission (%)')
plt.title('Transmission vs Frequency')
plt.grid(True)
plt.show()

In [ ]:
interference = []
for index in range(len(df)):
    freqs = np.array(df.iloc[index, 11])
    trans = np.array(df.iloc[index, 14])

    troguhs, props = find_peaks(-trans, prominence=1)
    widths_samples, _, left_ips, right_ips = peak_widths(-trans, troguhs, rel_height=0.75)

    left_freqs  = np.interp(left_ips,  np.arange(len(freqs)), freqs)
    right_freqs = np.interp(right_ips, np.arange(len(freqs)), freqs)
    widths_freq = right_freqs - left_freqs

    trough_freqs  = freqs[troguhs]
    trough_depths = props['prominences']

    trough_ranges = []
    for i in range(len(trough_freqs)):
        trough_ranges.append((trough_freqs[i] - widths_freq[i] / 2, trough_freqs[i] + widths_freq[i] / 2))

    trough_ranges = np.array(trough_ranges)

    closest_idxs = []

    for troguhs_range in trough_ranges:
        start, end = troguhs_range[0], troguhs_range[1]
        closest_start_idx = int(np.abs(freqs - start).argmin())
        closest_end_idx = int(np.abs(freqs - end).argmin())
        closest_idxs.append((closest_start_idx, closest_end_idx))

    interference.append(closest_idxs)

In [ ]:
index = 300
freqs = np.array(df.iloc[index, 11])
trans = np.array(df.iloc[index, 14])
# coeffs = np.polyfit(freqs, trans, 2)
# poly = np.poly1d(coeffs)
# fitted = poly(freqs)

plt.figure(figsize=(8,5))
plt.plot(range(len(freqs)), trans, 'o', label='Data')
# plt.plot(freqs, fitted, '-', label='Degree-3 fit')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Transmission (%)')
plt.title('Quadratic Fit to Transmission')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
peaks_df = pd.read_parquet(f'{cosmicai_path}/all_peaks_widths_heights.gzip')

In [ ]:
peaks  = peaks_df['Peaks (GHz)'].values
widths = peaks_df['Widths'].values
peak_ranges = np.column_stack((peaks - widths / 2, peaks + widths / 2))
peak_ranges_mid_points = np.array([float(start + end) / 2 for start, end in peak_ranges])

In [ ]:
freq_ranges = np.array([
    [float(min(x)), float(max(x))]
    for x in df['frequency_array'].tolist()
])
freq_ranges_mid_points = np.array([float(start + end) / 2 for start, end in freq_ranges])

In [ ]:
for idx, freq_points in enumerate(freq_ranges_mid_points):
    closest_idx = int(np.abs(peak_ranges_mid_points - freq_points).argmin())
    freqs = freq_ranges[idx]
    peaks = peak_ranges[closest_idx]
    start = max(freqs[0], peaks[0])
    end   = min(freqs[1], peaks[1])

In [ ]:
df['atmospheric_interference'] = interference

In [ ]:
df.to_csv(
    f'{cosmicai_path}/bandpass_augmented.csv',
    index=False
)

# Debugging

In [ ]:
cosmicai_path = 'Data'

df_scored = pd.read_parquet(f'{cosmicai_path}/bandpass_qa0_no_partitions_labelled_filt_scored.parquet')

In [ ]:
df_scored.info()

In [ ]:
df_scored['win_span'] = ((df_scored['win_end'] - df_scored['win_start']).abs() * (df_scored['frequency_array'].str[1] - df_scored['frequency_array'].str[0])).abs()

In [ ]:
df_scored.info()

In [ ]:
df_scored_filtered_special = df_scored[df_scored['win_span'] == 31250000.0]

In [ ]:
df_sorted_special = (
    df_scored_filtered_special
    .assign(freq_len = df_scored_filtered_special['frequency_array'].str.len())
    .sort_values(
        by=[ 'freq_len', 'score' ],
        ascending=[ True,    False ]
    )
    .drop(columns='freq_len')
    .reset_index(drop=True)
)

In [ ]:
per_fig = 10
out_dir = 'Images/Special'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
for length, grp in df_sorted_special.groupby(
        df_sorted_special['frequency_array'].str.len()
    ):
    n = len(grp)
    n_figs = math.ceil(n / per_fig)
    chunk = grp.iloc[fig_i*per_fig:(fig_i+1)*per_fig]
    fig_k = len(chunk)
    for fig_i in range(n_figs):
        chunk = grp.iloc[fig_i*per_fig:min((fig_i+1)*per_fig, len(grp))]
        fig_k = len(chunk)
        fig, axes = plt.subplots(fig_k, 1, figsize=(10, 3*fig_k))
        if fig_k == 1:
            axes = [axes]
        for ax, row in zip(axes, chunk.itertuples()):
            spec = row.amplitude_corr_tsys
            a, b = row.win_start, row.win_end
            atm_interfs = ast.literal_eval(row.atmospheric_interference)

            if len(atm_interfs) != 0:
                for item in atm_interfs:
                    c, d = item[0], item[1]
                    ax.axvspan(c, d, color='C9', alpha=0.2)

            x = np.arange(len(spec))
            ax.plot(x, spec, color='C0', label="Actual")

            ax.axvspan(a, b, color='C1', alpha=0.3)

            ax.set_title(
                f"eb_uid={row.eb_uid}, antenna={row.antenna}, polarization={row.polarization}, spw_name_ms={row.spw_name_ms}, score={row.score:.2f}, Range=[{a},{b}]"
            )
            ax.set_xlabel("Channel")
            ax.set_ylabel("Amplitude")
            ax.legend()
        plt.tight_layout(rect=[0,0,1,0.92])
        plt.suptitle(f"Items {fig_i*per_fig+1}–{fig_i*per_fig+fig_k}.", y=0.98)
        outpath = os.path.join(out_dir, f"length_{length}_fig{fig_i+1}.png")
        plt.savefig(outpath, dpi=500, bbox_inches="tight")
        plt.close(fig)
        if fig_i == 9:
            break

# Result Exploration

In [ ]:
df_scored = pd.read_parquet(f'science_scored.parquet')

In [ ]:
df_scored.info()

In [ ]:
PRIORITY = {
    "bandpass_amplitude_frequency",
    "bandpassflag_amplitude_frequency"        # ← add more as needed
}

def qa2_flag_code(flags):
    """
    Return 0 / 1 / 2  (no flags / priority flag / other flag).

    * priority = any list element whose lower-case *contains* a PRIORITY token
    * robust to lists, tuples, numpy arrays, or stringified lists
    """
    import ast, numpy as np, pandas as pd

    # ---- normalise to Python list ---------------------------------
    if flags is None or (isinstance(flags, float) and pd.isna(flags)):
        lst = []
    elif isinstance(flags, (list, tuple)):
        lst = list(flags)
    elif isinstance(flags, np.ndarray):
        lst = flags.tolist()
    else:
        s = str(flags).strip()
        if s in ("", "[]", "None"):
            lst = []
        else:                                # try to parse "['flag', ...]"
            try:
                parsed = ast.literal_eval(s)
                lst = list(parsed) if isinstance(parsed, (list, tuple, np.ndarray)) else [s]
            except Exception:
                lst = [s]

    # clean + lower-case for comparison
    clean = [str(f).strip().lower() for f in lst]

    # ---- classify -------------------------------------------------
    if not clean:
        return 0

    if any(any(token in f for token in PRIORITY) for f in clean):
        return 1
    return 2

In [ ]:
df_scored["qa2flag"] = df_scored["QA2 Flag(s)"].apply(qa2_flag_code)

In [ ]:
bins = np.round(np.arange(0.0, 1.0001, 0.05), 2)
labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins)-1)]

In [ ]:
df_scored = df_scored.copy()
df_scored["score_bin"] = pd.cut(df_scored["score"], bins=bins, labels=labels, include_lowest=True, right=True)

counts = (df_scored.loc[df_scored["qa2flag"] == 0]
            .groupby("score_bin")
            .size()
            .reindex(labels, fill_value=0))

totals = df_scored.groupby("score_bin").size().reindex(labels, fill_value=0)
rates = (counts / totals).fillna(0)

In [ ]:
counts

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
counts.plot(kind="bar", ax=ax, rot=90)
ax.set_xlabel("Score bin")
ax.set_ylabel("Count with qa2flag = 0")
ax.set_title("QA2=0 counts by 0.05 score bins")
ax.bar_label(ax.containers[0], labels=[str(int(v)) for v in counts.values], padding=3)
plt.tight_layout()
plt.show()

In [ ]:
df_scored_false_negative = df_scored[(df_scored['qa2flag'] == 1) & (df_scored['score'] <= 0.3)]

In [ ]:
df_scored_false_negative.info()

In [ ]:
df_scored_false_negative = df_scored_false_negative.copy()

In [ ]:
df_scored_false_negative.drop(columns=['score', 'win_start', 'win_end', 'score_bin'], inplace=True)

In [ ]:
df_scored_false_negative.to_parquet('Data/false_negative.parquet')

In [ ]:
def plot_df(df_slice, per_fig=10, k=None, out_dir="Images/false_negative_anaysis"):
    df_slice = df_slice.copy()
    df_slice['orig_idx'] = df_slice.index
    df_slice['atmospheric_interference'] = df_slice['atmospheric_interference'].apply(ast.literal_eval)
    df_slice = df_slice.rename(columns={"QA2 Flag(s)": "QA2_Flags"})


    n_figs = math.ceil(len(df_slice) / per_fig)
    for fig_i in range(n_figs):
        chunk = df_slice.iloc[fig_i * per_fig : (fig_i + 1) * per_fig]
        fig_k = len(chunk)
        fig, axes = plt.subplots(fig_k, 1, figsize=(10, 3 * fig_k))
        if fig_k == 1:
            axes = [axes]

        for ax, row in zip(axes, chunk.itertuples(index=False)):
            spec = np.array(row.amplitude_corr_tsys)
            buffer = spec.shape[0] // 20
            a, b = row.win_start, row.win_end
            
            if len(row.atmospheric_interference) != 0:
                for (c, d) in row.atmospheric_interference:
                    ax.axvspan(c, d, color='C9', alpha=0.2)

            x = np.arange(len(spec))
            ax.plot(x, spec, color='C0', label="Actual")

            if buffer > 0:
                ax.axvspan(0, buffer-1, color='gray', alpha=0.2)
                ax.axvspan(len(spec)-buffer, len(spec)-1, color='gray', alpha=0.2)

            ax.axvspan(a, b, color='C1', alpha=0.3)

            ax.set_title(
                f"eb_uid={row.eb_uid}, antenna={row.antenna}, spw_name_ms={row.spw_name_ms}, polarization={row.polarization}, score={row.score:.2f}, range=[{a},{b}], qa0 flags={row.QA2_Flags}"
            )
            ax.set_xlabel("Channel")
            ax.set_ylabel("Amplitude")
            ax.legend()

        plt.tight_layout(rect=[0, 0, 1, 0.92])
        plt.suptitle(
            f"Items {fig_i * per_fig + 1}–{fig_i * per_fig + fig_k} of Top {k}.",
            y=0.98
        )
        outpath = os.path.join(out_dir, f"top_{k}_fig{fig_i+1}.png")
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        plt.close(fig)

In [ ]:
plot_df(
    df_scored_false_negative,
    per_fig=8,
    k=len(df_scored_false_negative)
)

# Dataset Creation

In [ ]:
ROOT = Path("Data/variable_length_kernel")
OUT  = "Data/spotcheck.csv"
SEED = 42
ORIGINAL_PATH = Path("Data/bandpass_qa0_no_partitions_labelled_filt.parquet")

In [ ]:
TARGET_LENGTHS = {64, 120, 128, 240, 256, 480, 960, 1024, 1920, 2048}

In [ ]:
def load_original() -> pd.DataFrame:
    if 'df_original' in globals() and isinstance(df_original, pd.DataFrame):
        return df_original
    if ORIGINAL_PATH.suffix == ".parquet":
        return pd.read_parquet(ORIGINAL_PATH)
    elif ORIGINAL_PATH.suffix == ".csv":
        return pd.read_csv(ORIGINAL_PATH)
    raise ValueError("Set ORIGINAL_PATH to a .parquet or .csv, or define df_original in memory.")


In [ ]:
def infer_length_from_dir(p: Path) -> int:
    m = re.search(r"length_(\d+)", str(p))
    if not m:
        raise ValueError(f"Cannot infer length from: {p}")
    return int(m.group(1))

In [ ]:
def filter_original_by_length(df: pd.DataFrame, L: int) -> pd.DataFrame:
    if "length" in df.columns:
        return df[df["length"] == L]
    for col in ("amplitude_corr_tsys", "spec_arrays"):
        if col in df.columns:
            return df[df[col].map(len) == L]
    raise KeyError("Could not infer length—provide a 'length' column or an array-like column.")

In [ ]:
def pick_top10(score_df: pd.DataFrame) -> pd.DataFrame:
    if "score" in score_df.columns:
        top10 = score_df.sort_values("score", ascending=False, kind="mergesort").head(10).copy()
    else:
        top10 = score_df.head(10).copy()
    top10['frequency_array'] = top10['frequency_array'].apply(lambda s: np.array(ast.literal_eval(s), dtype=float))
    top10['frequency_array'] = top10['frequency_array'].apply(lambda freqs: [f*1e9 for f in freqs])
    top10["sample_group"] = "top10"
    return top10

In [ ]:
def pick_random10_from_original(orig_len_df: pd.DataFrame, exclude_uids: set) -> pd.DataFrame:
    pool = orig_len_df[~orig_len_df["uid"].isin(exclude_uids)]
    k = min(10, len(pool))
    out = pool.sample(n=k, random_state=SEED, replace=False).copy()
    out["sample_group"] = "random_original"
    return out

In [ ]:
df_full = load_original()
if "uid" not in df_full.columns:
    df_full = df_full.reset_index().rename(columns={"index": "uid"})

samples = []

In [ ]:
for length_dir in ROOT.glob("length_*"):
    L = infer_length_from_dir(length_dir)
    csvs = sorted(length_dir.glob("*.csv"))
    if not csvs:
        continue
    score_path = csvs[0]  # or choose by name if you have a convention
    score_df = pd.read_csv(score_path)

    if "uid" not in score_df.columns:
        raise KeyError(f"'uid' missing in {score_path}")

    top10 = pick_top10(score_df)

    orig_len_df = filter_original_by_length(df_full, L)
    rand10 = pick_random10_from_original(orig_len_df, exclude_uids=set(top10["uid"]))

    samples.append(pd.concat([top10, rand10], ignore_index=True))

In [ ]:
final = pd.concat(samples, ignore_index=True)
final.drop(columns=['uid', 'transmission_array', 'atmospheric_interference', 'score', 'kernel_size', 'win_start', 'win_end', 'sample_group'], inplace=True)

In [ ]:
final.info()

In [ ]:
final.to_csv(OUT, index=False, sep='|')